# Hierarchical Clustering: From First Principles

Hierarchical clustering builds a nested sequence of groups, from individual observations to one cluster containing all observations. The result is a hierarchy, not just one flat partition; a flat clustering can be obtained later by choosing where to cut the hierarchy.

This lesson focuses on agglomerative hierarchical clustering, the commonly used bottom-up version. Its merge decisions depend on a distance metric and a linkage rule, so those choices determine what kinds of groups the hierarchy favors.

## Why Use a Hierarchy?

Some analyses need to show relationships at several levels rather than return only one chosen number of clusters. For example, products might form broad categories that split into increasingly specific subgroups. A hierarchy records these nested decisions in one structure.

Unlike K-Means, agglomerative clustering does not require a cluster count before the merge process begins. The user can inspect the resulting dendrogram and select a cut afterward. This flexibility does not make the answer unique: distance metric, linkage, preprocessing, and the chosen cut all affect the result.

## Agglomerative Clustering

Agglomerative clustering starts with $n$ observations as $n$ singleton clusters. It then repeats the following process:

1. Compute the distance between each pair of current clusters using the selected linkage.
2. Merge the pair with the smallest linkage distance.
3. Update the distances from the new cluster to the remaining clusters using the linkage rule.
4. Continue until all observations are in one hierarchy, or stop earlier if a chosen threshold or cluster count is requested.

Each merge joins two existing groups and records the height at which they joined. The algorithm is greedy: a merge is not undone later. Therefore, an early local decision constrains all subsequent merges.

Divisive hierarchical clustering works in the opposite direction: it starts with one cluster and repeatedly splits it. It is less commonly used in standard library workflows; the remaining notes focus on the agglomerative procedure.

## Dendrogram and Cutting the Hierarchy

A dendrogram is a tree diagram of the agglomerative merge sequence. Leaves at the bottom represent observations. Each internal join shows which clusters merged, and its vertical height records the linkage distance or merge cost at that step.

A horizontal cut through the dendrogram defines a flat clustering: branches below the cut belong together, while branches separated above the cut remain distinct. A cut at a larger height generally produces fewer clusters; a lower cut produces more.

Large jumps in merge height can suggest a useful cut because they indicate that the next merge joins relatively dissimilar groups. This is a diagnostic, not a definitive rule. Dendrogram heights depend on the metric and linkage, and the analyst still needs to judge whether the resulting groups are useful.

## Linkage: Distance Between Clusters

A linkage rule defines how to measure distance between two clusters $A$ and $B$ from the distances between their observations. Different rules can produce different trees from the same data.

- **Single linkage:** the minimum pairwise distance, $d(A,B)=\min_{a\in A,b\in B}d(a,b)$. It can join long, irregular chains but is vulnerable to "chaining" through intermediate points.
- **Complete linkage:** the maximum pairwise distance, $d(A,B)=\max_{a\in A,b\in B}d(a,b)$. It favors compact clusters but can be sensitive to distant observations.
- **Average linkage:** the mean of all pairwise distances between observations in the two clusters. It is a compromise between single and complete linkage.
- **Ward linkage:** merges the pair that gives the smallest increase in within-cluster sum of squares. It favors compact groups and is defined for Euclidean geometry.

Linkage is not just a tuning detail: it defines what "closest clusters" means at every merge. The resulting dendrogram heights are interpreted according to the selected linkage.

## Example: Linkage Changes the Second Merge

Place three one-dimensional observations at $A=0$, $B=2$, and $C=5$, using absolute distance. All four linkage rules first merge $A$ and $B$, since their distance is 2, smaller than the other pairwise distances.

After that merge, the cluster is $\{A,B\}$. Its distance to $C$ differs by linkage:

| Linkage | Distance from $\{A,B\}$ to $C$ |
|---|---:|
| Single | $\min(5,3)=3$ |
| Average | $(5+3)/2=4$ |
| Complete | $\max(5,3)=5$ |

So even though the first merge is identical, the dendrogram's second merge height depends on the linkage definition. Ward linkage uses the increase in within-cluster squared error instead of these pairwise extrema or averages.

## Worked Ward Merge and Dendrogram Cuts

Place three one-dimensional points at $A=0$, $B=2$, and $C=5$. Ward linkage chooses the merge that produces the smallest increase in within-cluster sum of squares (WSS). For clusters $U$ and $V$, that increase is:

$$
\Delta\mathrm{WSS}(U,V)=\frac{|U||V|}{|U|+|V|}\left\|\boldsymbol{\mu}_U-\boldsymbol{\mu}_V\right\|_2^2
$$

Initially, the three candidate merge costs are:

| Pair | Increase in WSS |
|---|---:|
| $A,B$ | $\frac{1}{2}(0-2)^2=2$ |
| $B,C$ | $\frac{1}{2}(2-5)^2=4.5$ |
| $A,C$ | $\frac{1}{2}(0-5)^2=12.5$ |

Ward first merges $A$ and $B$, creating cluster $\{0,2\}$ with size 2 and mean 1. Its next merge with $C$ costs $\frac{2\cdot1}{3}(1-5)^2=32/3\approx10.67$. If we use increase in WSS as the height for this hand example, a cut below 2 gives three clusters, a cut between 2 and 10.67 gives $\{0,2\}$ and $\{5\}$, and a cut above 10.67 gives one cluster.

Some libraries display Ward's linkage height as $\sqrt{2\Delta\mathrm{WSS}}$ rather than $\Delta\mathrm{WSS}$. The merge order is unchanged, but use the library's height convention when selecting a numeric cut.

## Strengths, Limitations, and Practical Use

Hierarchical clustering provides a nested view of similarity and lets the user inspect several possible cluster counts from one merge tree. It can be useful for exploratory work on small or medium-sized datasets when the hierarchy itself is informative.

Its result depends on feature scaling, the distance metric, linkage, and cut height. Single linkage can chain points into elongated groups; complete and Ward tend to favor more compact groups. Once clusters merge, the greedy procedure does not undo the choice. Unlike a centroid model, an agglomerative hierarchy also does not automatically define a simple rule for assigning future unseen observations.

The method commonly requires pairwise distances, so memory use is often quadratic in the number of observations, and time can be quadratic or worse depending on the implementation and linkage. This can make it impractical for very large datasets.

Before clustering, choose a distance that matches the feature types and scale numeric features appropriately. Inspect a dendrogram and cluster stability rather than choosing a cut solely from a visual elbow. If using a flat partition, evaluate it with suitable internal or external measures and domain context; no single score establishes that a hierarchy is meaningful.

Use hierarchical clustering when nested group structure and post hoc cluster selection are valuable. Consider DBSCAN when noise and density-connected shapes matter, or K-Means when a scalable centroid-based partition with a chosen $K$ is appropriate.